# 06 - Cross-Domain Demo

End-to-end: NLU + multi-agent execution across Food, Instamart, Dineout.


In [ ]:
from bgi_trident.nlu.intent import classify_intent
from bgi_trident.nlu.entities import extract_entities
from bgi_trident.nlu.codemix import normalize_codemix, detect_language

text = "Meghana's biryani order maaDu, Harpic nu add maaDu, Saturday ge table book maaDu"
print(f'Language: {detect_language(text)}')
norm = normalize_codemix(text)
print(f'Normalized: {norm}')
print(f'Intent: {classify_intent(norm)}')
entities = extract_entities(norm)
print(f'Restaurants: {entities.restaurants}, Products: {entities.products}, Date: {entities.date}')


In [ ]:
import asyncio
from bgi_trident.orchestrator.coordinator import TridentCoordinator, AgentTask, IntentType
from bgi_trident.mcp.mock.food_mock import MockFoodMCP
from bgi_trident.mcp.mock.instamart_mock import MockInstamartMCP
from bgi_trident.mcp.mock.dineout_mock import MockDineoutMCP

async def demo():
    coord = TridentCoordinator(MockFoodMCP(), MockInstamartMCP(), MockDineoutMCP())
    await coord.start_session('U0042', 'kn')
    tasks = [
        AgentTask(agent_name='food', intent=IntentType.FOOD_ORDER, params={'tool': 'search_restaurants', 'query': 'Meghana Foods'}),
        AgentTask(agent_name='instamart', intent=IntentType.INSTAMART_PURCHASE, params={'tool': 'search_products', 'query': 'Harpic'}),
        AgentTask(agent_name='dineout', intent=IntentType.DINEOUT_BOOKING, params={'tool': 'search_restaurants_dineout', 'query': 'South Indian'}),
    ]
    result = await coord.execute(tasks)
    print(f'Completed: {result.tasks_completed}')
    for k, r in result.results.items():
        print(f'  {k}: {r.data}')
    await coord.close()

await demo()
